In [ ]:
!pip install tensorflow_hub opencv-python

In [ ]:
!pip install moviepy

In [1]:
# ===============================
# 🎨 ADVANCED VIDEO STYLE TRANSFER (Interactive: multiple style images + aspect selection)
# ===============================
import tensorflow_hub as hub
import tensorflow as tf
import cv2
import numpy as np
import urllib.request
from moviepy.editor import VideoFileClip, AudioFileClip
import os
import sys

# -------------------------------
# Load the TFHub style transfer model
# -------------------------------
print("🌀 Loading style transfer model...")
model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')
print("✅ Model loaded successfully!\n")

# -------------------------------
# Helpers: download if URL, load image (path or URL), blend styles
# -------------------------------
def download_if_url(url, filename=None):
    """Download file if url given. Returns local path."""
    if url.startswith("http"):
        if filename is None:
            filename = os.path.basename(url.split("?")[0]) or "downloaded_file"
        if not os.path.exists(filename):
            print(f"⬇️ Downloading {filename} ...")
            urllib.request.urlretrieve(url, filename)
            print(f"✅ Downloaded: {filename}")
        else:
            print(f"✅ {filename} already exists, skipping download.")
        return filename
    else:
        # local path
        if not os.path.exists(url):
            raise FileNotFoundError(f"File not found: {url}")
        return url

def load_image_tensor(path_or_url, target_size=None):
    """Load image from path or URL and return a float32 tensor [1,H,W,3] in [0,1]."""
    local_path = download_if_url(path_or_url) if path_or_url.startswith("http") else path_or_url
    img_bgr = cv2.imread(local_path)
    if img_bgr is None:
        raise ValueError(f"Cannot read image: {local_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if target_size:
        img_rgb = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_AREA)
    img_tf = tf.convert_to_tensor(img_rgb, dtype=tf.float32)
    img_tf = img_tf[tf.newaxis, ...] / 255.0
    return img_tf

def blend_style_images(style_paths, resize_to=(256, 256)):
    """Blend multiple style images by resizing them to resize_to and averaging."""
    tensors = []
    for p in style_paths:
        t = load_image_tensor(p, target_size=resize_to)  # [1,h,w,3]
        tensors.append(t)
    stacked = tf.concat(tensors, axis=0)  # [N,h,w,3]
    blended = tf.reduce_mean(stacked, axis=0, keepdims=True)  # [1,h,w,3]
    # normalize to [0,1] (should already be)
    blended = tf.clip_by_value(blended, 0.0, 1.0)
    return blended

# -------------------------------
# Style transfer helpers
# -------------------------------
def apply_style_transfer_to_frame(content_frame_rgb, style_tensor):
    """
    content_frame_rgb: HxWx3 uint8 RGB (0-255)
    style_tensor: tf tensor [1,sh,sw,3] float32 in [0,1]
    returns stylized_frame_rgb uint8 HxWx3
    """
    # resize content to model input 512x512 (or keep aspect by square crop; easier: resize)
    content_resized = cv2.resize(content_frame_rgb, (512, 512), interpolation=cv2.INTER_CUBIC)
    content_tf = tf.convert_to_tensor(content_resized, dtype=tf.float32)[tf.newaxis, ...] / 255.0  # [1,512,512,3]

    # run model
    stylized = model(tf.constant(content_tf), tf.constant(style_tensor))[0]  # [1,512,512,3]
    stylized = tf.clip_by_value(stylized, 0.0, 1.0)

    # optionally blend with original content to control strength (0.7 default)
    strength = 0.7
    blended = (1.0 - strength) * content_tf + strength * stylized  # [1,512,512,3]
    blended = tf.clip_by_value(blended, 0.0, 1.0)

    stylized_np = (blended[0].numpy() * 255.0).astype(np.uint8)  # HxWx3 RGB
    # resize back to original content_frame size
    stylized_resized_back = cv2.resize(stylized_np, (content_frame_rgb.shape[1], content_frame_rgb.shape[0]),
                                       interpolation=cv2.INTER_CUBIC)
    return stylized_resized_back

def color_correction_and_sharpen(stylized_rgb, original_rgb):
    """Apply simple color preservation and unsharp masking."""
    try:
        stylized_yuv = cv2.cvtColor(stylized_rgb, cv2.COLOR_RGB2YUV)
        original_yuv = cv2.cvtColor(original_rgb, cv2.COLOR_RGB2YUV)
        # Preserve luminance histogram from original
        stylized_yuv[:, :, 0] = cv2.equalizeHist(original_yuv[:, :, 0])
        corrected = cv2.cvtColor(stylized_yuv, cv2.COLOR_YUV2RGB)
    except Exception:
        corrected = stylized_rgb
    # sharpen
    gaussian = cv2.GaussianBlur(corrected, (0, 0), 3)
    sharpened = cv2.addWeighted(corrected, 1.5, gaussian, -0.5, 0)
    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    return sharpened

# -------------------------------
# Aspect ratio conversion helper
# -------------------------------
def convert_video_aspect_ratio(input_path, aspect_choice):
    """Converts video to many aspect ratios by center-cropping and resizing.
       Returns path to converted video."""
    aspect_ratios = {
        "16:9": (16, 9),
        "9:16": (9, 16),
        "1:1": (1, 1),
        "4:5": (4, 5),
        "5:4": (5, 4),
        "3:2": (3, 2),
        "2:3": (2, 3),
        "21:9": (21, 9),
        "9:21": (9, 21),
        "2:1": (2, 1)
    }

    if aspect_choice not in aspect_ratios:
        print("⚠️ Invalid choice. Skipping aspect ratio conversion.")
        return input_path

    target_ratio = aspect_ratios[aspect_choice][0] / aspect_ratios[aspect_choice][1]
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError("Cannot open video for conversion.")

    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0

    # Decide output resolution: keep width same, compute new height
    out_w = orig_w
    out_h = int(round(out_w / target_ratio))
    output_path = f"output_{aspect_choice.replace(':', '_')}.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        current_ratio = w / h

        if current_ratio > target_ratio:
            # too wide -> crop sides
            new_w = int(h * target_ratio)
            start_x = max((w - new_w) // 2, 0)
            frame_cropped = frame[:, start_x:start_x + new_w]
        elif current_ratio < target_ratio:
            # too tall -> crop top/bottom
            new_h = int(w / target_ratio)
            start_y = max((h - new_h) // 2, 0)
            frame_cropped = frame[start_y:start_y + new_h, :]
        else:
            frame_cropped = frame

        frame_resized = cv2.resize(frame_cropped, (out_w, out_h), interpolation=cv2.INTER_AREA)
        out.write(frame_resized)

    cap.release()
    out.release()
    print(f"✅ Converted video saved as: {output_path}")
    return output_path

# -------------------------------
# Main interactive flow
# -------------------------------
def main():
    try:
        video_input = input("🎥 Enter path or URL of the input video: ").strip()
        # prepare video local path
        if video_input.startswith("http"):
            video_path = download_if_url(video_input, filename="input_video.mp4")
        else:
            video_path = video_input
            if not os.path.exists(video_path):
                print("❌ Video path doesn't exist.")
                return

        # ask how many style images
        while True:
            try:
                n_styles = int(input("🖼️ How many style images do you want to use? (enter an integer >=1): ").strip())
                if n_styles >= 1:
                    break
            except ValueError:
                pass
            print("Please enter a valid integer >= 1.")

        style_paths = []
        for i in range(n_styles):
            s = input(f"👉 Enter path or URL for style image {i+1}: ").strip()
            style_paths.append(s)

        # load/blend styles
        if n_styles == 1:
            style_tensor = load_image_tensor(style_paths[0], target_size=(256,256))
            print("✅ Loaded single style image.")
        else:
            print("🔀 Blending style images...")
            style_tensor = blend_style_images(style_paths, resize_to=(256,256))
            print(f"✅ Blended {n_styles} style images.")

        # open video for processing
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("❌ Could not open video.")
            return

        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

        temp_output = "temp_stylized.mp4"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(temp_output, fourcc, fps, (orig_w, orig_h))

        frame_idx = 0
        print("\n🎨 Applying style transfer — processing frames (this may take time)...")
        while True:
            ret, frame_bgr = cap.read()
            if not ret:
                break
            # convert to RGB
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            # stylize
            stylized_rgb = apply_style_transfer_to_frame(frame_rgb, style_tensor)
            # color-correct & sharpen
            final_rgb = color_correction_and_sharpen(stylized_rgb, frame_rgb)
            # convert back to BGR for writing
            final_bgr = cv2.cvtColor(final_rgb, cv2.COLOR_RGB2BGR)
            out.write(final_bgr)

            frame_idx += 1
            if total_frames:
                print(f"Processed frame {frame_idx}/{total_frames}", end="\r")
            else:
                print(f"Processed frame {frame_idx}", end="\r")

        cap.release()
        out.release()
        print("\n\n✅ Style Transfer Completed! Temp stylized file:", temp_output)

        # merge original audio
        try:
            print("🎧 Merging with original audio...")
            video_clip = VideoFileClip(temp_output)
            audio_clip = AudioFileClip(video_path)
            final_output = "final_output_stylized_with_audio.mp4"
            final_clip = video_clip.set_audio(audio_clip)
            final_clip.write_videofile(final_output, codec="libx264", audio_codec="aac", bitrate="10000k", verbose=False, logger=None)
            print("✅ Merged audio. Output:", final_output)
        except Exception as e:
            print("⚠️ Failed to merge audio automatically:", str(e))
            final_output = temp_output
            print("Using video without merged audio:", final_output)

        # ask aspect ratio choice
        print("\n📏 Available aspect ratios:")
        print("16:9, 9:16, 1:1, 4:5, 5:4, 3:2, 2:3, 21:9, 9:21, 2:1")
        aspect_choice = input("👉 Enter desired aspect ratio (e.g., 16:9). Leave blank to skip: ").strip()
        if aspect_choice:
            converted = convert_video_aspect_ratio(final_output, aspect_choice)
            print(f"\n🎬 Final converted video: {converted}")
        else:
            print(f"\n🎬 Final video: {final_output}")

        print("\nAll done ✅")
    except KeyboardInterrupt:
        print("\nInterrupted by user. Exiting.")
    except Exception as exc:
        print("An error occurred:", exc)
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



🌀 Loading style transfer model...
✅ Model loaded successfully!

🎥 Enter path or URL of the input video: /content/sample_960x400_ocean_with_audio.mp4
🖼️ How many style images do you want to use? (enter an integer >=1): 1
👉 Enter path or URL for style image 1: /content/1024px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg
✅ Loaded single style image.

🎨 Applying style transfer — processing frames (this may take time)...
Processed frame 1116/1116

✅ Style Transfer Completed! Temp stylized file: temp_stylized.mp4
🎧 Merging with original audio...


  warnings.warn("Warning: in file %s, "%(self.filename)+



✅ Merged audio. Output: final_output_stylized_with_audio.mp4

📏 Available aspect ratios:
16:9, 9:16, 1:1, 4:5, 5:4, 3:2, 2:3, 21:9, 9:21, 2:1
👉 Enter desired aspect ratio (e.g., 16:9). Leave blank to skip: 4:5
✅ Converted video saved as: output_4_5.mp4

🎬 Final converted video: output_4_5.mp4

All done ✅
